In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

In [3]:
import os

REPO_DIR = '/content/seismic-gen'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ 클론 완료')
else:
    print('이미 존재함')

Cloning into '/content/seismic-gen'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 1), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (1/1), done.
✅ 클론 완료


In [4]:
# hifigan-dev 브랜치로 이동
!cd /content/seismic-gen && git fetch origin
!cd /content/seismic-gen && git checkout hifigan-dev
!cd /content/seismic-gen && git branch

Branch 'hifigan-dev' set up to track remote branch 'hifigan-dev' from 'origin'.
Switched to a new branch 'hifigan-dev'
* hifigan-dev
  main


In [5]:
!pip install h5py librosa torch torchaudio matplotlib numpy pandas seaborn scikit-learn -q

In [6]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# 경로
STEAD_HDF5 = '/content/drive/MyDrive/ML_Project/chunk3.hdf5'
STEAD_CSV  = '/content/drive/MyDrive/ML_Project/chunk3.csv'
SAVE_DIR   = '/content/drive/MyDrive/ML_Project/outputs/vae'
SPEC_DIR   = '/content/drive/MyDrive/ML_Project/outputs/spectrograms'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPEC_DIR, exist_ok=True)

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# 파일 존재 확인
print(f'hdf5: {os.path.exists(STEAD_HDF5)}')
print(f'csv:  {os.path.exists(STEAD_CSV)}')

device: cuda
hdf5: True
csv:  True


In [7]:
from sklearn.model_selection import train_test_split

# 메타데이터 로딩
metadata = pd.read_csv(STEAD_CSV, low_memory=False)
print(f'전체: {len(metadata):,}개')

# 학습/테스트 분리
train_meta = metadata[metadata['source_magnitude'] < 4.0].copy()
test_meta  = metadata[metadata['source_magnitude'] >= 4.0].copy().reset_index(drop=True)
train_meta, val_meta = train_test_split(train_meta, test_size=0.2, random_state=42)

print(f'train (M<4): {len(train_meta):,}개')
print(f'val   (M<4): {len(val_meta):,}개')
print(f'test  (M4+): {len(test_meta):,}개')

전체: 200,000개
train (M<4): 157,698개
val   (M<4): 39,425개
test  (M4+): 2,877개


In [8]:
import json

# CGM-GM 원본 STFT 설정
SR         = 100
N_FFT      = 160
HOP_LENGTH = 46
WIN_LENGTH = 160
N_FREQ     = N_FFT // 2 + 1  # 81
N_SAMPLES  = 6000
N_TIME     = N_SAMPLES // HOP_LENGTH + 1  # 131
LATENT_DIM = 16
N_COMP     = 3

print(f'N_FREQ : {N_FREQ}')
print(f'N_TIME : {N_TIME}')

# 조건 변수 (CGM-GM tcondvar=2 기준)
COND_COLS = [
    'source_magnitude',
    'source_latitude', 'source_longitude',
    'receiver_latitude', 'receiver_longitude',
    'source_depth_km'
]  # 6개 (angle 대신 src/sta 좌표 직접 사용)

# 조건 변수 정규화: min-max [0,1] (CGM-GM 원본 방식)
cond_min = train_meta[COND_COLS].min()
cond_max = train_meta[COND_COLS].max()

os.makedirs(SAVE_DIR, exist_ok=True)
norm_stats = {'min': cond_min.to_dict(), 'max': cond_max.to_dict()}
with open(os.path.join(SAVE_DIR, 'cond_norm_stats.json'), 'w') as f:
    json.dump(norm_stats, f, indent=2)

print(f'조건 변수 수: {len(COND_COLS)}')
print(pd.DataFrame({'min': cond_min, 'max': cond_max}))

N_FREQ : 81
N_TIME : 131
조건 변수 수: 6
                           min         max
source_magnitude     -0.420000    3.990000
source_latitude     -41.765800   71.197000
source_longitude   -173.073700  179.960200
receiver_latitude   -40.679647   70.340000
receiver_longitude -169.895100  176.492544
source_depth_km      -3.400000  304.000000


In [9]:
from torch.utils.data import Dataset, DataLoader
class STEADDataset(Dataset):
    # CGM-GM get_data.py 포팅 — STEAD HDF5 버전
    def __init__(self, meta_df, hdf5_path, cond_min, cond_max,
                 wfs_min=None, wfs_max=None, augment=False):
        self.meta      = meta_df.reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.cond_min  = torch.tensor(cond_min.values, dtype=torch.float32)
        self.cond_max  = torch.tensor(cond_max.values, dtype=torch.float32)
        self.wfs_min   = wfs_min  # log10 진폭 전체 min (train 기준)
        self.wfs_max   = wfs_max  # log10 진폭 전체 max (train 기준)
        self.augment   = augment
        self.hdf5_file = None
        self.window    = torch.hann_window(WIN_LENGTH)

    def _open_hdf5(self):
        if self.hdf5_file is None:
            self.hdf5_file = h5py.File(self.hdf5_path, 'r')

    def __len__(self):
        return len(self.meta)

    def _min_max_norm(self, x, x_min, x_max):
        # CGM-GM 원본 min_max_norm [0,1] sub
        return (x - x_min) / (x_max - x_min + 1e-10)

    def __getitem__(self, idx):
        self._open_hdf5()
        row        = self.meta.iloc[idx]
        trace_name = row['trace_name']

        # 파형 로딩
        try:
            wf = self.hdf5_file['data'][trace_name][:]  # (6000, 3)
        except KeyError:
            wf = np.zeros((N_SAMPLES, 3), dtype=np.float32)

        wf = wf[:N_SAMPLES].T.astype(np.float32)  # (3, 6000)

        # 3성분 평균 → 단일 성분 (CGM-GM은 단일 성분)
        wf_mono = wf.mean(axis=0, keepdims=True)  # (1, 6000)

        # STFT
        wf_tensor = torch.from_numpy(wf_mono)  # (1, 6000)
        stft      = torch.stft(
            wf_tensor[0],
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
            window=self.window, return_complex=True
        )  # (N_FREQ, N_TIME)

        # 위상/진폭 분리
        true_phase = torch.angle(stft)           # (N_FREQ, N_TIME)
        magnitude  = torch.abs(stft)             # (N_FREQ, N_TIME)

        # log10 변환 (CGM-GM 원본)
        log_amp = torch.log10(magnitude + 1e-10) # (N_FREQ, N_TIME)

        # permute → (N_TIME, N_FREQ)
        log_amp    = log_amp.permute(1, 0)        # (N_TIME, N_FREQ)
        true_phase = true_phase.permute(1, 0)     # (N_TIME, N_FREQ)

        # min-max 정규화 (wfs_min/max는 train 전체 기준)
        if self.wfs_min is not None:
            log_amp = self._min_max_norm(log_amp, self.wfs_min, self.wfs_max)

        # 조건 변수 min-max 정규화 (CGM-GM 원본)
        cond_vals = []
        for col in COND_COLS:
            v = row.get(col, 0.0)
            cond_vals.append(float(v) if not pd.isna(v) else 0.0)
        cond = torch.tensor(cond_vals, dtype=torch.float32)
        cond = self._min_max_norm(cond, self.cond_min, self.cond_max)

        # 원본 파형 (시간 도메인 손실용)
        orig_wf = torch.from_numpy(wf_mono[0])  # (6000,)
        mx = orig_wf.abs().max()
        if mx > 0:
          orig_wf = orig_wf / mx

        return log_amp, cond, true_phase, orig_wf


print('Dataset 클래스 정의 완료')

Dataset 클래스 정의 완료


In [10]:
# train 데이터 일부로 log10 진폭 min/max 추정
# CGM-GM 원본: 전체 데이터 기준이나 샘플로 근사
N_STAT_SAMPLES = 2000
sample_meta    = train_meta.sample(n=N_STAT_SAMPLES, random_state=42)
temp_ds        = STEADDataset(sample_meta, STEAD_HDF5,
                               cond_min, cond_max,
                               wfs_min=None, wfs_max=None)

all_log_amps = []
for i in range(len(temp_ds)):
    log_amp, _, _, _ = temp_ds[i]
    all_log_amps.append(log_amp)
    if (i+1) % 500 == 0:
        print(f'{i+1}/{N_STAT_SAMPLES}')

all_log_amps = torch.stack(all_log_amps)
wfs_min = all_log_amps.min()
wfs_max = all_log_amps.max()

print(f'wfs_min: {wfs_min:.4f}')
print(f'wfs_max: {wfs_max:.4f}')

500/2000
1000/2000
1500/2000
2000/2000
wfs_min: -10.0000
wfs_max: 7.0660


In [11]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 층화 추출
def stratified_sample(meta_df, n_samples, random_state=42):
    meta_df = meta_df.copy()
    meta_df['mag_bin'] = pd.cut(
        meta_df['source_magnitude'],
        bins=[-1, 1, 2, 3, 4],
        labels=['M<1', 'M1-2', 'M2-3', 'M3-4']
    )
    sampled = meta_df.groupby('mag_bin', observed=True).apply(
        lambda x: x.sample(
            n=max(1, int(n_samples * len(x) / len(meta_df))),
            random_state=random_state
        )
    ).reset_index(drop=True)
    return sampled.drop(columns=['mag_bin'])

train_meta_s = stratified_sample(train_meta, n_samples=30000)
val_meta_s   = stratified_sample(val_meta,   n_samples=6000)

print(f'train: {len(train_meta_s):,}개')
print(f'val:   {len(val_meta_s):,}개')
print('\ntrain 분포:')
print(pd.cut(train_meta_s['source_magnitude'],
             bins=[-1,1,2,3,4]).value_counts().sort_index())

BATCH_SIZE = 64

train_ds = STEADDataset(train_meta_s, STEAD_HDF5,
                        cond_min, cond_max, wfs_min, wfs_max, augment=True)
val_ds   = STEADDataset(val_meta_s,   STEAD_HDF5,
                        cond_min, cond_max, wfs_min, wfs_max)
test_ds  = STEADDataset(test_meta,    STEAD_HDF5,
                        cond_min, cond_max, wfs_min, wfs_max)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

log_amp_b, cond_b, phase_b, wf_b = next(iter(train_loader))
print(f'\nlog_amp : {log_amp_b.shape}')
print(f'cond    : {cond_b.shape}')
print(f'phase   : {phase_b.shape}')
print(f'wf      : {wf_b.shape}')

/tmp/ipykernel_4560/565850362.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = meta_df.groupby('mag_bin', observed=True).apply(
/tmp/ipykernel_4560/565850362.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = meta_df.groupby('mag_bin', observed=True).apply(


train: 29,998개
val:   5,998개

train 분포:
source_magnitude
(-1, 1]    11687
(1, 2]     12935
(2, 3]      4166
(3, 4]      1210
Name: count, dtype: int64

log_amp : torch.Size([64, 131, 81])
cond    : torch.Size([64, 6])
phase   : torch.Size([64, 131, 81])
wf      : torch.Size([64, 6000])


In [23]:
import sys
sys.path.insert(0, '/content/cgm-gm')
sys.path.insert(0, '/content/cgm-gm/model')
from dvae import cVAE
print('✅ 완료')

✅ 완료


In [25]:
import sys
sys.path.insert(0, '/content/cgm-gm')
sys.path.insert(0, '/content/cgm-gm/model')
from dvae import cVAE

FFT_SIZE = N_FFT        # 160
Z_DIM    = 16
NCOND    = 128
IN_SIZE  = len(COND_COLS)  # 6

model = cVAE(
    in_dim  = FFT_SIZE,
    z_dim   = Z_DIM,
    ncond   = NCOND,
    in_size = IN_SIZE
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'파라미터 수: {n_params:,}')

x   = log_amp_b[:2].to(device)
c   = cond_b[:2].to(device)
out, mu, lv = model(x, c)
print(f'입력 : {x.shape}')
print(f'복원 : {out.shape}')
print(f'mu   : {mu.shape}')

파라미터 수: 463,313
입력 : torch.Size([2, 131, 81])
복원 : torch.Size([2, 131, 81])
mu   : torch.Size([2, 131, 16])


In [26]:
import torchaudio
import torch.nn as nn

class TimeSpecConverter:
    # 시간-주파수 도메인 변환 (CGM-GM 원본)
    def __init__(self, n_fft, w_len, h_len, device, n_iter=50):
        self.n_fft  = n_fft
        self.w_len  = w_len
        self.h_len  = h_len
        self.device = device
        self.griffinlim = torchaudio.transforms.GriffinLim(
            n_fft=n_fft, n_iter=n_iter,
            win_length=w_len, hop_length=h_len, power=1.0
        ).to(device)

    def spec_to_time(self, wfs):
        # 복소 스펙트럼 → 파형
        window = torch.hann_window(self.w_len).to(wfs.device)
        return torch.istft(wfs, n_fft=self.n_fft,
                           hop_length=self.h_len, win_length=self.w_len,
                           window=window)

time_spec_converter = TimeSpecConverter(
    n_fft=N_FFT, w_len=WIN_LENGTH, h_len=HOP_LENGTH, device=device
)


def cgmgm_loss(X, X_hat, z_post_mean, z_post_logvar,
               z_prior_mean, z_prior_logvar,
               true_phase, time_domain_wfs,
               beta=1.0, alpha=1.0):
    # CGM-GM 원본 손실함수
    batch_size = X.shape[0]
    MSE = nn.MSELoss(reduction='sum')

    # 진폭 스펙트럼 재구성 손실
    L_recon = MSE(X_hat, X) / batch_size

    # 시간 도메인 손실 (CGM-GM 원본 방식)
    # min-max 역정규화
    wfs_hat_denorm = X_hat * (wfs_max - wfs_min) + wfs_min  # (B, T, F)
    eps = 1e-10
    # (B, T, F) → (B, F, T) → 진폭만 추출
    wfs_hat_perm = (torch.pow(10, wfs_hat_denorm) - eps).permute(0, 2, 1)
    magnitude_hat = torch.abs(wfs_hat_perm)  # get_phase_mag 중 magnitude만

    # true_phase: (B, T, F) → (B, F, T)
    phase_perm = true_phase.permute(0, 2, 1).to(magnitude_hat.device)

    wf_hat = time_spec_converter.spec_to_time(
        magnitude_hat * torch.exp(1j * phase_perm)
    )

    # time_domain_wfs: (B, 6000) 원본 파형과 비교
    min_len = min(wf_hat.shape[-1], time_domain_wfs.shape[-1])
    L_time  = MSE(
        wf_hat[..., :min_len],
        time_domain_wfs.to(wf_hat.device)[..., :min_len]
    ) / batch_size

    # KL divergence
    z_post_var  = torch.exp(z_post_logvar)
    z_prior_var = torch.exp(z_prior_logvar)
    L_KL = 0.5 * torch.sum(
        z_prior_logvar - z_post_logvar +
        ((z_post_var + torch.pow(z_post_mean - z_prior_mean, 2)) / z_prior_var) - 1
    ) / batch_size

    total = L_recon + beta * L_KL + alpha * L_time
    return total, {'recon': L_recon.item(),
                   'kl':    L_KL.item(),
                   'time':  L_time.item()}

print('손실함수 재정의 완료')

손실함수 재정의 완료


In [27]:
# 학습 데이터 5000개로 축소
train_meta_s = stratified_sample(train_meta, n_samples=5000)
val_meta_s   = stratified_sample(val_meta,   n_samples=1000)

train_ds = STEADDataset(train_meta_s, STEAD_HDF5,
                        cond_min, cond_max, wfs_min, wfs_max, augment=True)
val_ds   = STEADDataset(val_meta_s,   STEAD_HDF5,
                        cond_min, cond_max, wfs_min, wfs_max)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f'train: {len(train_loader)}배치')
print(f'val:   {len(val_loader)}배치')

train: 79배치
val:   16배치


/tmp/ipykernel_4560/565850362.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = meta_df.groupby('mag_bin', observed=True).apply(
/tmp/ipykernel_4560/565850362.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = meta_df.groupby('mag_bin', observed=True).apply(


In [6]:
#학습
import time
import torch.optim as optim

LR           = 3e-4
WEIGHT_DECAY = 1e-7
N_EPOCHS     = 30
BETA         = 1.0
ALPHA        = 1.0

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.99)

best_loss = float('inf')
history   = {'train': [], 'val': []}

print(f'학습 시작 — epochs={N_EPOCHS}, batch={BATCH_SIZE}, device={device}')
print('=' * 70)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_sub  = {'recon': 0, 'kl': 0, 'time': 0}
    t0 = time.time()

    for log_amp, cond, phase, wf in train_loader:
        log_amp = log_amp.to(device)
        cond    = cond.to(device)
        phase   = phase.to(device)
        wf      = wf.to(device)
        batch_size = log_amp.shape[0]

        log_amp_hat, z_post_mean, z_post_logvar = model(log_amp, cond)
        z_prior_mean, z_prior_logvar, _         = model.sample_prior(
            y=cond, n_sample=batch_size,
            seq_len=log_amp.shape[1],
            random_sampling=True, device=device
        )

        loss, sub = cgmgm_loss(
            log_amp, log_amp_hat,
            z_post_mean, z_post_logvar,
            z_prior_mean, z_prior_logvar,
            phase, wf, beta=BETA, alpha=ALPHA
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        for k in train_sub:
            train_sub[k] += sub[k]

    scheduler.step()
    n = len(train_loader)
    train_loss /= n
    for k in train_sub:
        train_sub[k] /= n
    history['train'].append(train_loss)

    # 검증
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for log_amp, cond, phase, wf in val_loader:
            log_amp = log_amp.to(device)
            cond    = cond.to(device)
            phase   = phase.to(device)
            wf      = wf.to(device)  # 추가
            batch_size = log_amp.shape[0]

            log_amp_hat, z_post_mean, z_post_logvar = model(log_amp, cond)
            z_prior_mean, z_prior_logvar, _         = model.sample_prior(
                y=cond, n_sample=batch_size,
                seq_len=log_amp.shape[1],
                random_sampling=True, device=device
            )
            loss, _ = cgmgm_loss(
                log_amp, log_amp_hat,
                z_post_mean, z_post_logvar,
                z_prior_mean, z_prior_logvar,
                phase, wf, beta=BETA, alpha=ALPHA
            )
            val_loss += loss.item()

    val_loss /= len(val_loader)
    history['val'].append(val_loss)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:3d}/{N_EPOCHS} | '
          f'train={train_loss:.4f} '
          f'(recon={train_sub["recon"]:.4f} '
          f'kl={train_sub["kl"]:.4f} '
          f'time={train_sub["time"]:.4f}) | '
          f'val={val_loss:.4f} | {elapsed:.1f}s')

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'val_loss':    val_loss,
        }, os.path.join(SAVE_DIR, 'vae_best.pth'))
        print(f'  ✅ best model 저장 (val={val_loss:.4f})')

NameError: name 'model' is not defined

In [28]:
ckpt = torch.load(os.path.join(SAVE_DIR, 'vae_best.pth'), map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'로드 완료 (epoch={ckpt["epoch"]}, val={ckpt["val_loss"]:.4f})')

로드 완료 (epoch=29, val=155.3290)


In [29]:
os.makedirs(SPEC_DIR, exist_ok=True)

def extract_spectrograms(loader, meta_df, split_name):
    # VAE 복원 스펙트럼 + 원본 파형 저장
    out_dir = os.path.join(SPEC_DIR, split_name)
    os.makedirs(out_dir, exist_ok=True)

    saved = 0
    with torch.no_grad():
        for log_amp, cond, phase, wf in loader:
            log_amp = log_amp.to(device)
            cond    = cond.to(device)

            recon, _, _ = model(log_amp, cond)

            for i in range(log_amp.shape[0]):
                trace_name = meta_df.iloc[saved]['trace_name']
                safe_name  = trace_name.replace('/', '_')

                # 복원 스펙트럼 (B, T, F) → (F, T)
                recon_np = recon[i].cpu().numpy().T          # (F, T)
                orig_np  = log_amp[i].cpu().numpy().T        # (F, T)
                wf_np    = wf[i].cpu().numpy()               # (6000,)
                phase_np = phase[i].cpu().numpy().T          # (F, T)

                np.save(os.path.join(out_dir, f'{safe_name}_recon.npy'),  recon_np)
                np.save(os.path.join(out_dir, f'{safe_name}_orig.npy'),   orig_np)
                np.save(os.path.join(out_dir, f'{safe_name}_wf.npy'),     wf_np)
                np.save(os.path.join(out_dir, f'{safe_name}_phase.npy'),  phase_np)

                saved += 1

            if saved % 500 == 0:
                print(f'[{split_name}] {saved}개 저장')

    print(f'✅ [{split_name}] 완료: {saved}개')

# val + test 추출
extract_spectrograms(val_loader,  val_meta_s,  'val')
extract_spectrograms(test_loader, test_meta,   'test')

✅ [val] 완료: 997개
✅ [test] 완료: 2877개


In [30]:
extract_spectrograms(train_loader, train_meta_s, 'train')

✅ [train] 완료: 4997개


In [33]:
!cp "/content/drive/MyDrive/Colab Notebooks/01_vae_spectrogram_extract.ipynb" \
    /content/seismic-gen/hifigan/01_vae_spectrogram_extract.ipynb

!cd /content/seismic-gen && git add hifigan/
!cd /content/seismic-gen && git commit -m "feat: VAE 학습 및 스펙트럼 추출 코드"
!cd /content/seismic-gen && git push origin hifigan-dev

cp: cannot create regular file '/content/seismic-gen/hifigan/01_vae_spectrogram_extract.ipynb': No such file or directory
fatal: pathspec 'hifigan/' did not match any files
On branch hifigan-dev
Your branch is up to date with 'origin/hifigan-dev'.

nothing to commit, working tree clean
Everything up-to-date
